## <a href="https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156170?b2cUser=true"><b>Langchain Retrieval</b></a><br/>

Utilizando para fazer pesquisas em documentos para responder perguntas 

<b>PASSOS:</b><br/>
<b>PASSO 1 - CARGA NO CARREGADOR</b><br/>
<b>PASSO 2 - CRIAÇÃO DO ÍNDICE DE BUSCA</b><br/>
<b>2.1 - QUEBRA DO TEXTO</b><br/>
<b>2.2 - INDEXANDO AS QUEBRAS DO TEXTO</b>

In [8]:
%pip install -qr requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#### <b>EXEMPLO 1</b><br/>
O objetivo será realizar a pesquisa em um arquivo txt, sobre os benefícios do cartão Gold contra roubo.

In [1]:
from langchain_openai import ChatOpenAI
from os import getenv
from dotenv import load_dotenv # CARREGA A VARIÁVEL DE AMBIENTE OPENAI_KEY LIDA DO ARQUIVO .env
from langchain_core.globals import set_debug

set_debug(True)

load_dotenv() # CARREGANDO O ARQUIVO COM A OPENAI_KEY

llm = ChatOpenAI( # INSTANCIANDO A LLM
                    model="gpt-5-mini",                    
                    # 1 - OBTENDO A API KEY POR MEIO DA VARIÁVEL DE AMBIENTE OPENAI_KEY. QUE VAI FICAR ARMAZENADA NO ARQUIVO .env.
                    # 2 - AINDA É NECESSÁRIO CARREGAR ESSE ARQUIVO. VER NA PRIMEIRA CÉLULA DO NOTEBOOK
                    api_key=getenv("OPENAI_KEY")                    
                )

#### <b>PASSO 1 - CARGA NO CARREGADOR</b>

Carregando

In [2]:
from langchain_community.document_loaders import TextLoader

# CRIAÇÃO
carregador = TextLoader("../documentos/GTB_gold_Nov23.txt",encoding="utf-8")

# CARGA NO CARREGADOR
documentos = carregador.load() # UM CARREGADOR DEVOLVE UM ARRAY DE DOCUMENTOS. NESSE CASO, EM PARTICULAR, SERÁ UM SÓ.

print('Documentos\n',documentos)

Documentos
 [Document(metadata={'source': '../documentos/GTB_gold_Nov23.txt'}, page_content='\n1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos

#### <b>PASSO 2 - CRIAÇÃO DO ÍNDICE DE BUSCA</b>

<li>Para isso, será necessário, primeiramente, realizar a quebra (splitter) em trechos, para que a IA possa indexá-los.

<b>2.1 - QUEBRA DO TEXTO</b>

In [3]:
from langchain_text_splitters import CharacterTextSplitter

# DEFINIÇÃO DO QUEBRADOR
quebrador = CharacterTextSplitter(chunk_size=1000) # QUEBRANDO EM CARACTERES. DE 1000 EM 1000 CARACTERES.

# QUEBRA DO TEXTO EM VÁRIOS TEXTOS
textos = quebrador.split_documents(documentos)

print('\nTextos\n',textos)


Textos
 [Document(metadata={'source': '../documentos/GTB_gold_Nov23.txt'}, page_content='1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  par

<b>2.3 - INDEXANDO AS QUEBRAS DE TEXTO</b>

<ol>
    <li>Sequências de palavras que possuem sentido semelhante, terão números semelhantes no espaço</li>
    <li>Esse números são chamados de índices, e esses <b>índices</b> recebem o nome de <b>embeddings</b></li>
</ol>

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# CRIANDO OS EMBEDDINGS
#embeddings = OpenAIEmbeddings(api_key=getenv("OPENAI_KEY"))
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

print('Embeddings\n',embeddings)

C:\Users\anton\AppData\Local\Temp\ipykernel_33544\3950281529.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

<b>2.4 - ARMAZENANDO OS ÍNDICES EM UM BANCO VETORIAL</b>

In [ ]:
from langchain.vectorstores import FAISS

# LUGAR PARA ARMAZENAR OS EMBEDDINGS -> BANCO VETORIAL. ARMAZENA O NÚMERO E A FRASE.
# SERÁ USADO O BANCO VETORIAL FAISS DO Facebook
FAISS.from_documents(textos,embeddings)   # CRIADO O BD A PARTIR DOS DOCUMENTOS
